In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv('churn-bigml-80.csv')
df.isnull().sum()

In [ ]:
x = df.drop(['Churn','State'],axis=1)
y = df['Churn'].copy
cat_cols = ['International plan','Voice mail plan']
preprocessor= ColumnTransformer(transformers=[(
    'cat',OneHotEncoder(handle_unknown='ignore', drop='first'),cat_cols
)], remainder = 'passthrough')
x_encoded=preprocessor.fit_transform(x)
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x_encoded)



In [ ]:
inertia = []
K_range = range(1,11)
for k in K_range:
  kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
  kmeans.fit(x_scaled)
  inertia.append(kmeans.inertia_)
plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker='o')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.grid(True)
plt.show()

In [ ]:
best_k = 3
kmeans = KMeans(n_clusters=best_k, random_state=42)
kmeans.fit(x_scaled)
df['cluster']= kmeans.labels_
print(df['cluster'].value_counts())

In [ ]:
sil = silhouette_score(x_scaled, kmeans.labels_)
print(f'silhouette_score:{sil:4f}')

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(x_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df['cluster'], palette='Set2', s=60)
plt.xlabel('PCA Component 1')
plt.ylabel('PCA Component 2')
plt.title(f'K-Means Clusters (K={best_k}) on Churn Dataset')
plt.legend(title='Cluster')
plt.show()

In [ ]:
print("\n=== Mean values per cluster (interpretation) ===")
print(df.groupby('cluster').mean(numeric_only=True).round(2))